# Connect to the LLM Gateway

The travel agent you deployed in notebook 02 is calling Bedrock directly. In this notebook you
will point it at the LLM Gateway from Module 2 so every model call flows through a governed
proxy with virtual keys, budgets, and cost tracking.

Steps:

1. Look up (or mint) the LLM Gateway URL and a virtual key
2. Publish them at `/workshop/llm-gateway-*` (shared) **and** `/FAST-stack/llm_gateway_*` (agent)
3. Touch `travel_agent.py` to force a container rebuild and `cdk deploy`
4. Verify — call `/chat/completions` directly and then check the virtual key's spend


## Step 1: Read or Create the LLM Gateway Credentials

If a previous user already minted a virtual key, reuse it. Otherwise mint a new one with a $10
budget. The helper stores both values at `/workshop/llm-gateway-url` and
`/workshop/llm-gateway-key` in SSM Parameter Store.


In [ ]:
import boto3, json, os, subprocess, urllib.request, urllib.error

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )

ssm = boto3.client("ssm", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
sm = boto3.client("secretsmanager", region_name=REGION)


def read_ssm(name):
    try:
        return ssm.get_parameter(Name=name)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        return ""


# The gateway URL is ALWAYS taken from the live stack output, never from a cached
# SSM value. A redeploy mints a new API Gateway with a new hostname, so a stale
# /workshop/llm-gateway-url from a prior run would point at a deleted endpoint and
# every call below would fail with a DNS error. The stack output is authoritative.
stack = cfn.describe_stacks(StackName="workshop-llm-gateway-stack")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}
LLM_GATEWAY_URL = outputs["ProxyUrl"].rstrip("/")

admin_key = sm.get_secret_value(SecretId="workshop-llm-gateway-stack-admin-key")["SecretString"]
if not admin_key.startswith("sk-"):
    admin_key = "sk-" + admin_key


def key_is_valid(key):
    """A cached virtual key is only reusable if it still resolves against THIS
    gateway. A key minted on a prior deployment is unknown to a fresh proxy."""
    if not key:
        return False
    try:
        req = urllib.request.Request(
            f"{LLM_GATEWAY_URL}/key/info",
            headers={"Authorization": f"Bearer {key}"},
            method="GET",
        )
        with urllib.request.urlopen(req, timeout=15):
            return True
    except urllib.error.HTTPError:
        return False
    except Exception:
        return False


LLM_GATEWAY_KEY = read_ssm("/workshop/llm-gateway-key")

if key_is_valid(LLM_GATEWAY_KEY):
    print("Reusing existing virtual key from /workshop/llm-gateway-key")
else:
    print("Minting a new LLM Gateway virtual key")
    req = urllib.request.Request(
        f"{LLM_GATEWAY_URL}/key/generate",
        data=json.dumps({
            "key_alias": "fast-agent-key",
            "max_budget": 10.0,
        }).encode(),
        headers={
            "Authorization": f"Bearer {admin_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        body = json.loads(resp.read().decode())
    LLM_GATEWAY_KEY = body["key"]
    ssm.put_parameter(Name="/workshop/llm-gateway-key", Value=LLM_GATEWAY_KEY, Type="String", Overwrite=True)

# Keep the shared URL parameter in sync with the authoritative stack output.
ssm.put_parameter(Name="/workshop/llm-gateway-url", Value=LLM_GATEWAY_URL, Type="String", Overwrite=True)

print(f"LLM Gateway URL: {LLM_GATEWAY_URL}")
print(f"Virtual Key:     (value not printed)")

## Step 2: Publish for the FAST Agent

`travel_agent.py` reads `/${STACK_NAME}/llm_gateway_url` and `/${STACK_NAME}/llm_gateway_key`
from SSM at startup. FAST's default stack name is `FAST-stack`.


In [ ]:
for name, value in [
    ("/FAST-stack/llm_gateway_url", LLM_GATEWAY_URL),
    ("/FAST-stack/llm_gateway_key", LLM_GATEWAY_KEY),
]:
    ssm.put_parameter(Name=name, Value=value, Type="String", Overwrite=True)
    print(f"Stored {name}")


## Step 3: Force a Container Rebuild

The AgentCore Runtime reads SSM parameters when the container starts. Appending a comment to
`travel_agent.py` changes the file hash, which makes CDK rebuild and redeploy the image.


In [ ]:
import pathlib, os, subprocess

FAST_DIR = pathlib.Path("/workshop/fast-agent")
TRAVEL_PY = FAST_DIR / "patterns" / "strands-travel-agent" / "travel_agent.py"

if not TRAVEL_PY.exists():
    raise RuntimeError(f"{TRAVEL_PY} missing — run notebook 02 first")

with TRAVEL_PY.open("a") as f:
    f.write("# LLM Gateway integration\n")
print(f"Touched {TRAVEL_PY}")

os.chdir(FAST_DIR / "infra-cdk")
r = subprocess.run(
    ["npx", "cdk", "deploy", "--require-approval", "never"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(
        f"cdk deploy failed (exit {r.returncode}).\n"
        f"stdout tail: {r.stdout[-2500:]}\nstderr tail: {r.stderr[-2500:]}"
    )
print(r.stdout[-600:])
print("Redeploy complete")

## Step 4: Verify the Gateway Works

Hit `/chat/completions` directly with the virtual key. If this returns text, the gateway and
key are functional and the agent will use them after the redeploy.


In [ ]:
import json, urllib.request

req = urllib.request.Request(
    f"{LLM_GATEWAY_URL}/chat/completions",
    data=json.dumps({
        "model": "claude-sonnet",
        "messages": [{"role": "user", "content": "Say hello in one word"}],
        "max_tokens": 10,
    }).encode(),
    headers={
        "Authorization": f"Bearer {LLM_GATEWAY_KEY}",
        "Content-Type": "application/json",
    },
    method="POST",
)
with urllib.request.urlopen(req, timeout=30) as resp:
    body = json.loads(resp.read().decode())

print("Model reply:", body["choices"][0]["message"]["content"])


## Step 5: Check Virtual-Key Spend

Once your agent sends a message, spend on this key should be > $0.

Open the Amplify URL (from notebook 02) and send a message such as "hello". Then re-run the
cell below — if `spend` increased, the agent is routing through the LLM Gateway.


In [ ]:
import json, urllib.request

req = urllib.request.Request(
    f"{LLM_GATEWAY_URL}/key/info",
    headers={"Authorization": f"Bearer {LLM_GATEWAY_KEY}"},
    method="GET",
)
with urllib.request.urlopen(req, timeout=15) as resp:
    info_wrapper = json.loads(resp.read().decode())

info = info_wrapper.get("info", info_wrapper)
spend = float(info.get("spend", 0))
budget = float(info.get("max_budget", 0))
print(f"Spend: ${spend:.4f} / ${budget:.2f} budget")

if spend > 0:
    print("The agent is routing through the LLM Gateway.")
else:
    print("Send a message in the Amplify UI, then re-run this cell.")


## Summary

The agent now calls Bedrock through Module 2's LLM Gateway. Next: wire the agent's tools to
the AgentCore Gateway from Module 3.

- If you followed Module 3a (OSS MCP Registry), continue with **notebook 04a**.
- If you followed Module 3b (AgentCore Registry), continue with **notebook 04b**.
